# House Price Prediction

**Goal:** Predict the sale price of residential homes
**Algorithm:** Gradient Boosting Regressor
**Dataset:** [House Prices Competition](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
%matplotlib inline

In [ ]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


## 1. Load Data from Kaggle

In [ ]:
path = kagglehub.competition_download("house-prices-advanced-regression-techniques")
train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
print ('Train shape: %s' % (train.shape,))
print ('Test shape: %s' % (test.shape,))
print ('\nTrain columns: %d features' % len(train.columns))

<hr>## 2. Exploratory Data Analysis

In [ ]:
print ('SalePrice statistics:\n%s' % train['SalePrice'].describe())
print ('\nSkewness: %.2f' % train['SalePrice'].skew())
print ('\nColumns with >50%% missing:')
missing = train.isnull().mean()
print (missing[missing > 0.5].to_string())

In [ ]:
# SalePrice distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(train['SalePrice'], bins=50, kde=True)
plt.title('SalePrice Distribution (skewed)')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(train['SalePrice']), bins=50, kde=True)
plt.title('Log-transformed SalePrice (normal)')

plt.tight_layout()
plt.show()
print ('After log transform, skewness: %.3f' % np.log1p(train['SalePrice']).skew())

In [ ]:
# Top correlations with SalePrice
numeric = train.select_dtypes(include=[np.number])
corrs = numeric.corr()['SalePrice'].sort_values(ascending=False)
print ('Top 10 features correlated with SalePrice:')
print (corrs.head(10))

<hr>## 3. Preprocessing

In [ ]:
def preprocess(df):
    data = df.copy()

    # Drop columns with >50% missing
    drop_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu']
    data = data.drop([c for c in drop_cols if c in data.columns], axis=1)

    # Fill numeric with median
    num_cols = data.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if col != 'Id':
            data[col] = data[col].fillna(data[col].median())

    # Fill & encode categoricals
    cat_cols = data.select_dtypes(include=['object']).columns
    for col in cat_cols:
        data[col] = data[col].fillna(data[col].mode()[0])
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col].astype(str))

    return data

train_proc = preprocess(train)
test_proc = preprocess(test)

# Align columns
common = [c for c in train_proc.columns if c in test_proc.columns]
X = train_proc[[c for c in common if c != 'SalePrice']]
y = np.log1p(train['SalePrice'])  # log transform
X_test = test_proc[[c for c in common if c != 'SalePrice']]

print ('Features after preprocessing: %d' % X.shape[1])
print ('Missing values remaining: %d' % (X.isnull().sum().sum() + y.isnull().sum().sum()))

<hr>## 4. Train/Test Split

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)
print ('Train: %d, Validation: %d' % (X_tr.shape[0], X_val.shape[0]))

<hr>## 5. Train Model

In [ ]:
model = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    min_samples_leaf=5, random_state=42)
model.fit(X_tr, y_tr)
print ('Gradient Boosting model trained')
print ('Number of estimators: %d' % model.n_estimators_)

<hr>## 6. Evaluate Performance

In [ ]:
y_val_pred_log = model.predict(X_val)
y_val_pred = np.expm1(y_val_pred_log)
y_val_actual = np.expm1(y_val)

rmse = np.sqrt(mean_squared_error(y_val_actual, y_val_pred))
r2 = r2_score(y_val_actual, y_val_pred)
rmsle = np.sqrt(mean_squared_error(y_val, model.predict(X_val)))

print ('Results (actual dollar values):')
print ('RMSE:  $%.2f' % rmse)
print ('R²:     %.4f' % r2)
print ('RMSLE:  %.4f (Kaggle uses this!)' % rmsle)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_val_actual, y_val_pred, alpha=0.5)
plt.plot([y_val_actual.min(), y_val_actual.max()],
         [y_val_actual.min(), y_val_actual.max()], 'r--', lw=2)
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.title('Actual vs Predicted House Prices (R² = %.4f)' % r2)
plt.tight_layout()
plt.show()

<hr>## 7. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print ('Top 10 most important features:')
print (importances.head(10).to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x='importance', y='feature',
            data=importances.head(10), palette='viridis')
plt.title('Top 10 Features for House Price Prediction')
plt.tight_layout()
plt.show()

<hr>## 8. Predict Test Set & Save Submission

In [ ]:
test_pred = np.expm1(model.predict(X_test))
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': test_pred
})
submission.to_csv('house_prices_submission.csv', index=False)

print ('Submission saved to house_prices_submission.csv')
print ('Price range: $%.0f - $%.0f' % (test_pred.min(), test_pred.max()))
print ('Mean price: $%.0f' % test_pred.mean())
print ('\nFirst 5 rows:\n%s' % submission.head())